<a href="https://colab.research.google.com/github/lizzie-qing/Avatar-Demo/blob/main/Dlib.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install dlib opencv-python pandas numpy


In [2]:
import zipfile
from pathlib import Path

BASE_DIR = Path("/content/poster_face_analysis")
INPUT_DIR = BASE_DIR / "02_cropped_faces"
CHECK_DIR = BASE_DIR / "03_landmark_check"
OUTPUT_DIR = BASE_DIR / "04_features"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

zip_path = "/content/cropped_faces_test.zip"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(INPUT_DIR)

print("解压完成")
print("图片数量：", len(list(INPUT_DIR.glob("*"))))

解压完成
图片数量： 38


In [3]:
!wget http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
!bzip2 -d shape_predictor_68_face_landmarks.dat.bz2

--2026-06-04 10:08:10--  http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
Resolving dlib.net (dlib.net)... 107.180.26.78
Connecting to dlib.net (dlib.net)|107.180.26.78|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2 [following]
--2026-06-04 10:08:10--  https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
Connecting to dlib.net (dlib.net)|107.180.26.78|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64040097 (61M)
Saving to: ‘shape_predictor_68_face_landmarks.dat.bz2’

shape_predictor_68_ 100%[===================>]  61.07M  15.9MB/s    in 3.8s    

2026-06-04 10:08:14 (15.9 MB/s) - ‘shape_predictor_68_face_landmarks.dat.bz2’ saved [64040097/64040097]



In [4]:
import cv2
import dlib
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path("/content/poster_face_analysis")

INPUT_DIR = BASE_DIR / "02_cropped_faces"
CHECK_DIR = BASE_DIR / "03_landmark_check"
OUTPUT_DIR = BASE_DIR / "04_features"

MODEL_PATH = Path("/content/shape_predictor_68_face_landmarks.dat")

detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor(str(MODEL_PATH))

def shape_to_np(shape, dtype="int"):
    coords = np.zeros((68, 2), dtype=dtype)
    for i in range(68):
        coords[i] = (shape.part(i).x, shape.part(i).y)
    return coords

def draw_landmarks(image, landmarks):
    output = image.copy()
    for i, (x, y) in enumerate(landmarks):
        cv2.circle(output, (x, y), 2, (0, 0, 255), -1)
        cv2.putText(
            output,
            str(i + 1),
            (x + 2, y - 2),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.3,
            (255, 0, 0),
            1,
            cv2.LINE_AA
        )
    return output

def choose_main_face(faces):
    if len(faces) == 0:
        return None
    return max(faces, key=lambda rect: rect.width() * rect.height())

image_extensions = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
image_files = [
    f for f in INPUT_DIR.iterdir()
    if f.suffix.lower() in image_extensions
]

landmark_rows = []
log_rows = []

print(f"找到 {len(image_files)} 张图片，开始 dlib 检测...")

for image_path in sorted(image_files):
    filename = image_path.name
    poster_id = image_path.stem.replace("_face", "")

    image = cv2.imread(str(image_path))

    if image is None:
        log_rows.append({
            "poster_id": poster_id,
            "filename": filename,
            "status": "failed",
            "num_faces": 0,
            "problem": "cannot_read_image"
        })
        continue

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    faces = detector(gray, 1)
    num_faces = len(faces)

    if num_faces == 0:
        log_rows.append({
            "poster_id": poster_id,
            "filename": filename,
            "status": "failed",
            "num_faces": 0,
            "problem": "no_face_detected"
        })
        print(f"{filename}: failed")
        continue

    main_face = choose_main_face(faces)
    shape = predictor(gray, main_face)
    landmarks = shape_to_np(shape)

    row = {
        "poster_id": poster_id,
        "filename": filename,
        "num_faces_detected": num_faces,
        "image_width": image.shape[1],
        "image_height": image.shape[0],
        "face_left": main_face.left(),
        "face_top": main_face.top(),
        "face_right": main_face.right(),
        "face_bottom": main_face.bottom(),
    }

    for i, (x, y) in enumerate(landmarks, start=1):
        row[f"x{i}"] = x
        row[f"y{i}"] = y

    landmark_rows.append(row)

    log_rows.append({
        "poster_id": poster_id,
        "filename": filename,
        "status": "success",
        "num_faces": num_faces,
        "problem": ""
    })

    checked_image = draw_landmarks(image, landmarks)
    output_check_path = CHECK_DIR / f"{poster_id}_landmark_check.png"
    cv2.imwrite(str(output_check_path), checked_image)

    print(f"{filename}: success, faces={num_faces}")

landmarks_df = pd.DataFrame(landmark_rows)
log_df = pd.DataFrame(log_rows)

landmarks_csv = OUTPUT_DIR / "poster_dlib_landmarks_test.csv"
log_csv = OUTPUT_DIR / "poster_dlib_detection_log_test.csv"

landmarks_df.to_csv(landmarks_csv, index=False, encoding="utf-8-sig")
log_df.to_csv(log_csv, index=False, encoding="utf-8-sig")

print("完成")
print("总图片数:", len(image_files))
print("成功检测:", (log_df["status"] == "success").sum())
print("检测失败:", (log_df["status"] == "failed").sum())
print("landmark csv:", landmarks_csv)
print("log csv:", log_csv)

找到 37 张图片，开始 dlib 检测...
JP_MED_001_face.png: success, faces=1
JP_MED_002_face.png: success, faces=1
JP_MED_003_face.png: success, faces=1
JP_MED_004_face.png: success, faces=1
JP_MED_005_face.png: success, faces=1
JP_MED_006_face.png: success, faces=1
JP_MED_007_face.png: success, faces=1
JP_MED_008_face.png: success, faces=1
JP_MED_009_face.png: success, faces=1
JP_MED_010_face.png: success, faces=1
JP_MED_011_face.png: success, faces=1
JP_MED_012_face.png: success, faces=1
JP_MED_013_face.png: success, faces=1
JP_MED_014_face.png: success, faces=1
JP_MED_015_face.png: success, faces=1
JP_MED_016_face.png: success, faces=1
JP_MED_017_face.png: success, faces=1
JP_MED_018_face.png: success, faces=1
JP_MED_019_face.png: success, faces=1
JP_MED_020_face.png: success, faces=1
JP_MED_021_face.png: success, faces=1
JP_MED_022_face.png: success, faces=1
JP_MED_023_face.png: success, faces=1
JP_MED_024_face.png: success, faces=1
JP_MED_025_face.png: success, faces=1
JP_MED_026_face.png: succe